## 1. Hãy đọc dữ liệu từ các file csv, sử dụng tự suy ra kiểu dữ liệu cho mỗi cột. 

In [21]:
from pyspark.sql import SparkSession

# Init SparkSession
spark = SparkSession.builder.appName("Lab3").getOrCreate()

# read csv file
df_orders = spark.read.format("csv").options(header='True', delimiter=';', inferSchema='True').load("./data/Orders.csv")
df_customers = spark.read.format("csv").options(header='True', delimiter=';', inferSchema='True').load("./data/Customer_List.csv")
df_products = spark.read.format("csv").options(header='True', delimiter=';', inferSchema='True').load("./data/Products.csv")
df_order_items = spark.read.format("csv").options(header='True', delimiter=';', inferSchema='True').load("./data/Order_Items.csv")
df_reviews = spark.read.format("csv").options(header='True', delimiter=';', inferSchema='True').load("./data/Order_Reviews.csv")

## 2. Thống kê tổng số đơn hàng, số lượng khách hàng và người bán

In [22]:
# Total orders
count_orders = df_orders.count()
print("Total orders:", count_orders)

# Total customers
count_customers = df_customers.select("Customer_Trx_ID").distinct().count()
print("Total customers:", count_customers)

# Total sellers
count_sellers = df_order_items.select("Seller_ID").distinct().count()
print("Total sellers:", count_sellers)

Total orders: 99441
Total customers: 99442
Total sellers: 3095


## 3. Phân tích số lượng đơn hàng theo quốc gia, sắp xếp theo thứ tự giảm dần

In [23]:
# Join Orders và Customer_List to get conuntry information
df_orders_customers = df_orders.join(df_customers, on="Customer_Trx_ID", how="inner")

# Count and order by desc
orders_by_country = df_orders_customers.groupBy("Customer_Country").count().orderBy("count", ascending=False)
orders_by_country.show(10)

+----------------+-----+
|Customer_Country|count|
+----------------+-----+
|         Germany|41754|
|          France|12848|
|     Netherlands|11629|
|         Belgium| 5464|
|         Austria| 5043|
|     Switzerland| 3640|
|  United Kingdom| 3382|
|          Poland| 2139|
|         Czechia| 2034|
|           Italy| 2025|
+----------------+-----+
only showing top 10 rows


## 4. Phân tích số lượng đơn hàng nhóm theo năm, tháng đặt hàng

In [24]:
from pyspark.sql.functions import year, month

# group by year and month
orders_by_month_year = df_orders.withColumn("year", year("Order_Purchase_Timestamp")) \
                          .withColumn("month", month("Order_Purchase_Timestamp"))

results = orders_by_month_year.groupBy("year", "month").count().sort("year", "month", ascending=[True, False])
results.show(10)

+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2022|   12|    1|
|2022|   10|  324|
|2022|    9|    4|
|2023|   12| 5673|
|2023|   11| 7544|
|2023|   10| 4631|
|2023|    9| 4285|
|2023|    8| 4331|
|2023|    7| 4026|
|2023|    6| 3245|
+----+-----+-----+
only showing top 10 rows


## 5. Thống kê điểm đánh giá trung bình, số lượng đánh giá theo từng mức

In [25]:
from pyspark.sql.functions import col

df_reviews = df_reviews.filter(col("Review_Score").isin(["1", "2", "3", "4", "5"]))

df_reviews = df_reviews.withColumn("Review_Score", col("Review_Score").cast("int"))

# calculate average review score
avg_review = df_reviews.groupBy().avg("Review_Score").first()[0]
print("Avg point:", avg_review)

count_review = df_reviews.groupBy("Review_Score").count().orderBy("Review_Score", ascending=False)
count_review.show(10)

Avg point: 4.0864214950162765
+------------+-----+
|Review_Score|count|
+------------+-----+
|           5|57328|
|           4|19141|
|           3| 8179|
|           2| 3151|
|           1|11424|
+------------+-----+



## 7. Xác định sản phẩm có số lượng bán ra cao nhất và tính điểm đánh giá trung bình cho từng sản phẩm 

In [33]:
from pyspark.sql.functions import count, first, avg

# Join Orders with Order_Items
df_order_join_items = df_order_items.join(
    df_orders.select("Order_ID"),
    on="Order_ID",
    how="inner"
)

# Join products
df_product_join_order_items = df_order_join_items.join(
    df_products.select("Product_ID", "Product_Category_Name"),
    on="Product_ID",
    how="inner"
)

# Join with df_previews 
df_product_item_reviews = df_product_join_order_items.join(
    df_reviews.select("Order_ID", "Review_Score"),
    on="Order_ID",
    how="inner"
)

# Group by Product_ID and find max counting products, get name of product
counting_products = df_product_item_reviews.groupBy("Product_ID") \
    .agg(
        count("*").alias("count_products"),
        first("Product_Category_Name").alias("Product_Name")
    ) \
    .orderBy("count_products", ascending=False)

# Get first product
total_selled, name_ID = counting_products.first()[1], counting_products.first()[0]
print(f"Product with ID {name_ID} has been sold the most with total {total_selled} times.")

# Cal avg review score for each product category
avg_review_by_category = df_product_item_reviews.groupBy("Product_ID") \
    .agg(
        avg("Review_Score").alias("Avg_Review_Score-Products"),
        first("Product_Category_Name").alias("Product_Name")
    ) \
    .orderBy("Avg_Review_Score-Products", ascending=False)

avg_review_by_category.show(10)

Product with ID aca2eb7d00ea1a7b8ebd4e68314663af has been sold the most with total 524 times.
+--------------------+-------------------------+-------------------+
|          Product_ID|Avg_Review_Score-Products|       Product_Name|
+--------------------+-------------------------+-------------------+
|0030026a6ddb3b2d1...|                      5.0|     Sports_Leisure|
|0036bb031e69d915c...|                      5.0|       Garden_Tools|
|00126f27c81360368...|                      5.0|         Cool_Stuff|
|003128f981470c3e5...|                      5.0|     Sports_Leisure|
|001b237c0e9bb435f...|                      5.0|     Bed_Bath_Table|
|004552d98c5d3653a...|                      5.0|    Furniture_Decor|
|001c5d71ac6ad696d...|                      5.0|     Bed_Bath_Table|
|003962cb74a8b43cf...|                      5.0|Luggage_Accessories|
|002ec297b1b00fb9d...|                      5.0|         Housewares|
|000d9be29b5207b54...|                      5.0|      Watches_Gifts|
+--------